starting with how machines learn linear regression

In [1]:
import math
import random
random.seed(10)

## A Probabilistic Interpretation of Least Squares

### Model

We assume

$$
y = w x + \varepsilon
$$

where $\varepsilon$ is noise modelled by a random normal variable:

$$
\varepsilon \sim \mathcal{N}(0, \sigma^2)
$$

So each target is normally distributed around the prediction:

$$
p(y_i \mid x_i, w) = \mathcal{N}(y_i \mid x_i w, \sigma^2)
$$

In words: the probability that $y$ takes this value, given $x$ and $w$, is the probability of that value under a normal distribution whose mean is the prediction $x_i w$ and whose variance is $\sigma^2$.

### Likelihood

The likelihood of $w$ is the probability of the observed $y$ under that $w$:

$$
L(w) = \mathcal{N}(y_i \mid x_i w, \sigma^2)
$$

For a whole dataset (and a weight vector $\mathbf{w}$), take the product over all data points:

$$
L(\mathbf{w}) = \prod_{i} \mathcal{N}(y_i \mid \mathbf{x}_i^\top \mathbf{w}, \sigma^2)
$$

### Maximum likelihood

We want to maximise $L(\mathbf{w})$, or equivalently any strictly increasing function of it, such as $\log L(\mathbf{w})$.

- Write out the normal density (in terms of $e$ and $\pi$), with mean $\mathbf{x}_i^\top \mathbf{w}$ and standard deviation $\sigma$.
- Take the log and discard every term that doesn't vary with $\mathbf{w}$, since $\mathbf{w}$ is the only thing we can change.

$$
\log L(\mathbf{w}) = \text{const} - \frac{1}{2\sigma^2} \sum_i \left(y_i - \mathbf{x}_i^\top \mathbf{w}\right)^2
$$

### Result

Maximising $\log L(\mathbf{w})$ is the same as minimising

$$
\sum_i \left(y_i - \mathbf{x}_i^\top \mathbf{w}\right)^2
$$

which is **least squares**.

In [2]:
# neuron class
class neuron:
    def __init__(self, weight, bias = 0):
        self.weight = weight
        self.bias = bias

# formula linking energy to projectile range
def projectile_range(energy,starting_distance=0):
    r = round(2*energy / 9.81,5)
    return r + starting_distance

In [3]:
# generating data
random_integers = [random.randint(1,100) for _ in range(10000)]
random_distances = list(map(projectile_range,random_integers))
for energy, distance in list(zip(random_integers, random_distances))[:10]:
    print(energy, distance,round(distance/energy,4))

# learning function without accounting for bias
def learning_v1(neuron,input,correct,lr=0.1):
    output = neuron.weight * input
    error = output - correct
    if error > 0:
        neuron.weight -= lr
    else:
        neuron.weight += lr

single_neuron = neuron(1)
for energy,distance in list(zip(random_integers,random_distances)):
    learning_v1(single_neuron,energy,distance,0.001)
    # print(single_neuron.weight)
print(round(single_neuron.weight,4),round(single_neuron.bias))

74 15.08665 0.2039
5 1.01937 0.2039
55 11.21305 0.2039
62 12.64016 0.2039
74 15.08665 0.2039
2 0.40775 0.2039
27 5.50459 0.2039
60 12.23242 0.2039
63 12.84404 0.2039
36 7.33945 0.2039
0.204 0


In [4]:
def learning_v2(neuron,input,correct,lr = 0.1):
    output = neuron.weight * input + neuron.bias
    error = output - correct
    if error > 0:
        neuron.weight -= lr * input
        neuron.bias -= lr 
    else:
        neuron.weight += lr * input
        neuron.bias += lr 
    # print(neuron.weight,energy,neuron.weight)
bias = random.randint(0,50)
# bias = 4
print(bias)

random_integers = [random.randint(1,100) for _ in range(1000000)]
random_distances = list(map(projectile_range,random_integers))
for energy, distance in list(zip(random_integers, random_distances))[:10]:
    print(energy, distance, bias, round(distance + bias, 4), round(distance/energy,4))

29
16 3.26198 29 32.262 0.2039
62 12.64016 29 41.6402 0.2039
4 0.81549 29 29.8155 0.2039
1 0.20387 29 29.2039 0.2039
26 5.30071 29 34.3007 0.2039
18 3.66972 29 32.6697 0.2039
68 13.8634 29 42.8634 0.2039
16 3.26198 29 32.262 0.2039
41 8.35882 29 37.3588 0.2039
63 12.84404 29 41.844 0.2039


In [5]:
biased_neuron = neuron(1,1)
for energy,distance in list(zip(random_integers,random_distances)):
    learning_v2(biased_neuron,energy,distance+bias,0.0001)
print(round(biased_neuron.weight,4),round(biased_neuron.bias,4))

0.205 29.0008


In [6]:
# basically everything in the notebook:
#  the neuron weight is changed with respect to input as its effect on the error is reflective of that
# the bias only changes based on the learning rate: its changes shouldnt depend on input as its impact on the error here do not depend on input

def learning_v3(neuron,input,correct,lr = 0.1):
    output = neuron.weight * input + neuron.bias
    error = output - correct

    neuron.weight -= lr * input * error
    neuron.bias -= lr * error

# little cleaner: error will already have its direction and we want to make changes opposite to that direction

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

# Data (y ≈ 2x + 1)
X = torch.tensor([[1.],
                  [2.],
                  [3.],
                  [4.]])
y = torch.tensor([[3.],
                  [5.],
                  [7.],
                  [9.]])

# Model: linear regression = single linear layer
model = nn.Linear(1, 1)

# Loss: Mean Squared Error
criterion = nn.MSELoss()

# Optimiser: gradient descent
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Training loop
for _ in range(1000):
    optimizer.zero_grad()

    y_hat = model(X)        # forward pass
    loss = criterion(y_hat, y)

    loss.backward()         # backprop
    optimizer.step()        # update weights

print("weight:", model.weight.item())
print("bias:", model.bias.item())


weight: 2.001112699508667
bias: 0.9967294931411743
